# 랜덤신호론 13주차 실습
## KNN 랜덤신호 분류모델

1. 회귀와 분류를 구분한다.
2. KNN의 이웃과 k를 설명한다.
3. 표준화와 KNN을 파이프라인으로 연결한다.
4. 정확도와 혼동행렬로 모델을 평가한다.


## 0. 공통 환경 준비


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

rng=np.random.default_rng(2026)
plt.style.use('seaborn-v0_8-whitegrid')
FEATURE_NAMES=['mean','std','peak_to_peak','rms','autocorrelation','dominant_frequency','spectral_energy']

def lag1_autocorrelation(x):
    x=np.asarray(x,float)
    if len(x)<2 or np.std(x[:-1])==0 or np.std(x[1:])==0: return 0.0
    return float(np.corrcoef(x[:-1],x[1:])[0,1])

def extract_features(x,fs):
    x=np.asarray(x,float); centered=x-x.mean()
    spectrum=np.abs(np.fft.rfft(centered)); freqs=np.fft.rfftfreq(len(x),1/fs)
    return {'mean':float(x.mean()),'std':float(x.std()),'peak_to_peak':float(np.ptp(x)),
      'rms':float(np.sqrt(np.mean(x**2))),'autocorrelation':lag1_autocorrelation(x),
      'dominant_frequency':float(freqs[np.argmax(spectrum)]),
      'spectral_energy':float(np.sum(spectrum**2)/len(x))}

def generate_signal(kind,fs=50,seconds=4,seed=0):
    local=np.random.default_rng(seed); t=np.arange(0,seconds,1/fs)
    clean=np.sin(2*np.pi*3*t)+.35*np.sin(2*np.pi*7*t)
    if kind=='normal': x=clean+local.normal(0,.18,len(t))
    elif kind=='high_noise': x=clean+local.normal(0,.75,len(t))
    elif kind=='spike':
        x=clean+local.normal(0,.18,len(t)); idx=local.choice(len(t),4,False); x[idx]+=local.choice([-5,5],4)
    else: raise ValueError('kind 확인')
    return t,x

def build_dataset(n=40,fs=50):
    rows=[]; labels=['normal','high_noise','spike']
    for j,label in enumerate(labels):
        for i in range(n):
            _,x=generate_signal(label,fs=fs,seed=1000+100*j+i)
            row=extract_features(x,fs); row['label']=label; rows.append(row)
    return pd.DataFrame(rows)
print('준비 완료')


## 1. 데이터 준비


In [ ]:
fs=50;df=build_dataset(60,fs);X=df[FEATURE_NAMES];y=df.label
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)


## 2. KNN 학습


In [ ]:
model=make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=5))
model.fit(X_train,y_train);y_pred=model.predict(X_test)
print('정확도:',accuracy_score(y_test,y_pred))


**관찰 1** StandardScaler와 KNN의 역할을 설명하세요.


## 3. 혼동행렬


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test,y_pred,cmap='Blues');plt.show()
print(classification_report(y_test,y_pred,zero_division=0))


## 4. k 비교


In [ ]:
scores=[]
for k in [1,3,5,7,11,21]:
    m=make_pipeline(StandardScaler(),KNeighborsClassifier(k));m.fit(X_train,y_train)
    scores.append((k,accuracy_score(y_test,m.predict(X_test))))
print(scores);plt.plot([x[0] for x in scores],[x[1] for x in scores],marker='o');plt.show()


**미니 과제** 가장 높은 결과의 k와 이 결과만으로 k를 확정할 때의 한계를 설명하세요.


## 참고 자료
- KNeighborsClassifier: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html
- Pipeline: https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html
- confusion_matrix: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html
